# 🔧 02 — Transformación
**Metodología HEFESTO — Paso 3: Modelo Lógico del DW**

Lee los datos de `02_interim/`, aplica limpieza y construye las tablas del esquema Estrella:
`dimTiempo`, `dimProducto`, `dimGeografia`, `dimCanal` y `factVentas`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from src.transform import (
    cargar_config,
    limpiar_dataframe,
    construir_dim_tiempo,
    construir_dim_producto,
    construir_dim_geografia,
    construir_dim_canal,
    construir_fact_ventas,
    guardar_tablas_procesadas,
)

## 1️⃣ Leer datos intermedios

In [ ]:
df_raw = pd.read_parquet("../data/02_interim/01_raw_cargado.parquet")
print(f"Filas cargadas: {len(df_raw):,}")
df_raw.head(3)

## 2️⃣ Limpieza y normalización

In [ ]:
config = cargar_config("../config/settings.yaml")
df_clean = limpiar_dataframe(df_raw, config)
df_clean.head(3)

In [ ]:
# Verificar que no queden nulos problemáticos
df_clean.isnull().sum()[df_clean.isnull().sum() > 0]

## 3️⃣ Construir dimensiones (HEFESTO — Paso 3.2)

In [ ]:
dim_tiempo = construir_dim_tiempo(df_clean)
print("\ndimTiempo:")
dim_tiempo

In [ ]:
dim_producto = construir_dim_producto(df_clean)
print(f"\ndimProducto ({len(dim_producto)} filas):")
dim_producto.head(10)

In [ ]:
dim_geografia = construir_dim_geografia(df_clean)
print(f"\ndimGeografia ({len(dim_geografia)} filas):")
dim_geografia.head(10)

In [ ]:
dim_canal = construir_dim_canal(df_clean)
print("\ndimCanal:")
dim_canal

## 4️⃣ Construir tabla de hechos (HEFESTO — Paso 3.3)

Indicadores:
- `cantidad_vendida`   = SUM(qty) donde status != Cancelled
- `monto_total`        = SUM(amount) donde status != Cancelled
- `cantidad_cancelada` = SUM(qty) donde status == Cancelled
- `monto_cancelado`    = SUM(amount) donde status == Cancelled

In [ ]:
fact_ventas = construir_fact_ventas(
    df_clean, dim_tiempo, dim_producto, dim_geografia, dim_canal, config
)
fact_ventas.head(10)

In [ ]:
# Validación: totales deben coincidir con el CSV original
print("Total unidades vendidas:", fact_ventas["cantidad_vendida"].sum())
print("Total monto vendido (INR):", fact_ventas["monto_total"].sum():,.2f)
print("Total unidades canceladas:", fact_ventas["cantidad_cancelada"].sum())

## 5️⃣ Guardar en `03_processed/`

In [ ]:
guardar_tablas_procesadas(
    dim_tiempo, dim_producto, dim_geografia, dim_canal, fact_ventas, config
)
print("\n✅ Tablas guardadas en data/03_processed/")

## ✅ Resultado

Todas las tablas del DW fueron construidas y guardadas como `.parquet` en `03_processed/`.

▶️ Siguiente paso: corré `03_load.ipynb`